# Week 6 - 03: Query the Vector Database
Now that our documents are stored in ChromaDB, we will ask questions and retrieve the most relevant chunks.

Example:
> "How can AI help with timetable scheduling?"

The important point is that we do **not** search only for exact words.
We search using embeddings, so similar meaning can be found.


In [ ]:
!pip install chromadb sentence-transformers

In [2]:
import chromadb
from sentence_transformers import SentenceTransformer

# Load the same embedding model used when storing the documents.
model = SentenceTransformer("all-MiniLM-L6-v2")

# Open the same local ChromaDB folder.
client = chromadb.PersistentClient(path="./chroma_week6")

# Open the collection we created earlier.
collection = client.get_collection(name="week6_documents")

print("Items in database:", collection.count())


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Items in database: 8


In [ ]:
question = "How can AI be used to create a timetable?"

# Convert the question into an embedding.
question_embedding = model.encode(question).tolist()

print("Question:", question)
print("Embedding size:", len(question_embedding))


Question: How can AI be used to create a timetable?
Embedding size: 384


In [ ]:
# Query ChromaDB.
# n_results=3 means:
# "Give me the 3 most similar documents."

results = collection.query(
    query_embeddings=[question_embedding],
    n_results=3
)

print(results["documents"])

[['A timetable scheduling system can use AI to assign teachers, rooms, courses, and time slots.', 'Artificial Intelligence allows computers to perform tasks that normally require human intelligence.', 'Machine learning is a part of AI where computers learn patterns from data.']]


In [ ]:
# The result is nested because Chroma supports multiple queries.
# results["documents"][0] means the results for our first question.

for i, document in enumerate(results["documents"][0]):
    print(f"\nResult {i + 1}:")
    print(document)


Result 1:
A timetable scheduling system can use AI to assign teachers, rooms, courses, and time slots.

Result 2:
Artificial Intelligence allows computers to perform tasks that normally require human intelligence.

Result 3:
Machine learning is a part of AI where computers learn patterns from data.


In [6]:
# Chroma also gives distance values.
# Smaller distance generally means the vectors are more similar.

print("Distances:")
for distance in results["distances"][0]:
    print(distance)


Distances:
0.43929189443588257
1.0262365341186523
1.118389368057251


In [7]:
# Let's make a reusable search function.

def semantic_search(question, number_of_results=3):
    # Step 1: convert question into an embedding
    question_embedding = model.encode(question).tolist()

    # Step 2: search the vector database
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=number_of_results
    )

    # Step 3: print the matching documents
    print(f"\nQuestion: {question}")
    print("-" * 60)

    for i, document in enumerate(results["documents"][0]):
        distance = results["distances"][0][i]
        print(f"\nResult {i + 1} | distance = {distance:.4f}")
        print(document)

    return results


In [8]:
# Try different questions.

semantic_search("What is semantic search?")
semantic_search("How can scheduling use constraints?")
semantic_search("What does a genetic algorithm do?")



Question: What is semantic search?
------------------------------------------------------------

Result 1 | distance = 0.4913
Semantic search finds information based on meaning, not only exact keywords.

Result 2 | distance = 1.1826
Natural language processing helps computers work with human language such as text and speech.

Result 3 | distance = 1.2736
Vector databases store embeddings and make similarity search fast.

Question: How can scheduling use constraints?
------------------------------------------------------------

Result 1 | distance = 0.5558
Constraint satisfaction problems are useful when a scheduling problem has many rules.

Result 2 | distance = 1.0860
A timetable scheduling system can use AI to assign teachers, rooms, courses, and time slots.

Result 3 | distance = 1.5348
Artificial Intelligence allows computers to perform tasks that normally require human intelligence.

Question: What does a genetic algorithm do?
-----------------------------------------------------

{'ids': [['doc_7', 'doc_0', 'doc_1']],
 'embeddings': None,
 'documents': [['Genetic algorithms can search for good solutions by using selection, crossover, and mutation.',
   'Artificial Intelligence allows computers to perform tasks that normally require human intelligence.',
   'Machine learning is a part of AI where computers learn patterns from data.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None, None, None]],
 'distances': [[0.7142148017883301, 1.1282051801681519, 1.1608084440231323]]}

## What you learned

The complete search flow is:

**Question**
→ **Question embedding**
→ **Vector database**
→ **Similarity search**
→ **Most relevant chunks**

This is the main practical task for Week 6.
